In [ ]:
import json
from typing import List, Dict
from groq import Groq
from tqdm import tqdm


class GroqClient:
    """
    Wrapper cho Groq API
    """

    def __init__(
        self,
        api_key: str,
        model: str = "llama-3.3-70b-versatile",
        temperature: float = 0.0,
    ):
        self.client = Groq(api_key=api_key)
        self.model = model
        self.temperature = temperature

    def generate(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=[
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"}
        )

        return response.choices[0].message.content


class HashtagCache:
    """
    Lưu mapping hashtag trước và sau chỉnh sửa
    """

    def __init__(self):
        self.map: Dict[str, str] = {}

    def add(self, original: str, normalized: str):
        self.map[original] = normalized

    def exists(self, hashtag: str) -> bool:
        return hashtag in self.map

    def get(self, hashtag: str) -> str:
        return self.map[hashtag]

    def save(self, path: str):

        with open(path, "w", encoding="utf8") as f:
            json.dump(self.map, f, ensure_ascii=False, indent=2)

    def load(self, path: str):

        with open(path, "r", encoding="utf8") as f:
            self.map = json.load(f)


class HashtagNormalizer:
    """
    Normalize hashtags using LLM
    """

    SYSTEM_PROMPT = """
You normalize TikTok hashtags.

Rules:
- remove #
- split words
- restore Vietnamese diacritics if needed
- keep lowercase
- output JSON only

Example:
input:
["gaixinh","bestfoodvn"]

* Note: just answer the json in output
output:
{
 "gaixinh":"gái xinh",
 "bestfoodvn":"best food vn"
}
"""

    def __init__(
        self,
        llm: GroqClient,
        cache: HashtagCache,
        batch_size: int = 50,
    ):
        self.llm = llm
        self.cache = cache
        self.batch_size = batch_size

    def _build_prompt(self, hashtags: List[str]):

        hashtags = [h.replace("#", "") for h in hashtags]

        return f"""
{self.SYSTEM_PROMPT}

input:
{json.dumps(hashtags)}

output:
"""

    def _call_llm(self, hashtags: List[str]):

        prompt = self._build_prompt(hashtags)

        response = self.llm.generate(prompt)

        try:
            data = json.loads(response)
        except Exception:
            raise ValueError("LLM output not valid JSON:\n" + response)

        return data

    def normalize(self, hashtags: List[str]) -> Dict[str, str]:

        results = {}

        to_process = []

        for h in hashtags:

            if self.cache.exists(h):
                results[h] = self.cache.get(h)
            else:
                to_process.append(h)

        for i in tqdm(range(0, len(to_process), self.batch_size)):

            batch = to_process[i : i + self.batch_size]

            data = self._call_llm(batch)

            for original, normalized in data.items():

                key = "#" + original if not original.startswith("#") else original

                self.cache.add(key, normalized)
                results[key] = normalized

        return results


# ----------------------------
# Example usage
# ----------------------------

if __name__ == "__main__":

    API_KEY = "API-KEY"

    llm = GroqClient(api_key=API_KEY)

    cache = HashtagCache()

    normalizer = HashtagNormalizer(
        llm=llm,
        cache=cache,
        batch_size=40,
    )

    # test_hashtags = [
    #     "#gaixinh",
    #     "#xuhuong2024",
    #     "#bestfoodvn",
    #     "#learnpython",
    #     "#thanhxuan",
    # ]

    # result = normalizer.normalize(test_hashtags)

    # print(result)

    # cache.save("hashtag_map.json")

In [17]:
hashtags = []

with open('/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/sentiment_analysist/output/filtered_unique_hashtags__20260312_203656.txt', "r", encoding="utf8") as f:
    for line in f:
        tag = line.strip()

        if tag == "":
            continue

        if not tag.startswith("#"):
            tag = "#" + tag

        hashtags.append(tag)
            
            
    result = normalizer.normalize(hashtags)

    # print(result)

    cache.save("output/hashtag_map.json")

  3%|▎         | 1/34 [00:01<00:51,  1.56s/it]

{
  "andem":"ăn đêm",
   "giaitri":"giải trí",
   "phuyen":"phù yên",
   "pets":"pets",
   "traidep":"trái dép",
   "vietnamtoiyeu":"việt nam tôi yêu",
   "kpopfyp":"kpop fyp",
   "studyasmr":"study asmr",
   "the100":"the 100",
   "racing":"racing",
   "pialinh":"pi linh",
   "xinh":"xinh",
   "dirtbike":"dirt bike",
   "adventuretime":"adventure time",
   "ruabien":"rửa biển",
   "taekwondo":"taekwondo",
   "restling":"restling",
   "jessi":"jessi",
   "ancungtiktok":"ăn cúng tiktok",
   "xyzcba":"xyz cba",
   "mma":"mma",
   "idontwanttobehere":"i dont want to be here",
   "christmastree":"christmas tree",
   "leavemealone":"leave me alone",
   "onechampionship":"one championship",
   "nonstop":"non stop",
   "hoathinh":"hóa tinh",
   "zevia":"zevia",
   "sunworld":"sun world",
   "memes":"memes",
   "hotmom":"hot mom",
   "sabaidilife":"sabai di life",
   "goride":"go ride",
   "tiktokgiaitri":"tiktok giải trí",
   "nature":"nature",
   "dedinsayda":"dedin sayda",
   "losangeles":"

  6%|▌         | 2/34 [00:02<00:40,  1.25s/it]

{
  "trying":"trying",
   "khh":"khh",
   "barclubmusic":"bar club music",
   "мотоспорт":"moto sport",
   "อาหาร":"อาหาร",
   "athlete":"athlete",
   "sadsong":"sad song",
   "depresion":"depression",
   "emo":"emo",
   "help":"help",
   "paintok":"paint ok",
   "sportsmanship":"sportsmanship",
   "holidays":"holidays",
   "fyp":"fyp",
   "visitorlando":"visitor londo",
   "meaningless":"meaningless",
   "depressionhub":"depression hub",
   "cutebaby":"cute baby",
   "portsantai":"port santai",
   "stitch":"stitch",
   "babycute":"baby cute",
   "hoathinhvuinhon":"hoa tinh vui hon",
   "studymotivation":"study motivation",
   "fashion":"fashion",
   "h2ofestival":"h2o festival",
   "foodtiktok":"food tiktok",
   "ansiedadenãoéfrescura":"ansiedade nao e frescura",
   "randomrestock":"random restock",
   "sadtimes":"sad times",
   "xhuong":"xhuong",
   "kyucngoisao":"kyuc ngoi sao",
   "coping":"coping",
   "bungy":"bungy",
   "bar":"bar",
   "bjj":"bjj",
   "tutorial":"tutorial",
   "e

  9%|▉         | 3/34 [00:03<00:37,  1.20s/it]

...

{
  "\u0111\u1ea7ub\u1ebfp":"đậu bắp",
   "break":"break",
   "striper":"striper",
   "\ub304\uc2a4":"모자",
   "hanhphuc":"hạnh phúc",
   "\u043c\u043e\u0442\u043e\u0446\u0438\u043a\u043b\u0438":"мотоцикли",
   "sadslideshow":"sad slideshow",
   "stationery":"stationery",
   "camping":"camping",
   "amazing":"amazing",
   "ngan98tiktok":"ngan 98 tiktok",
   "skezix":"skezix",
   "\u043c\u043e\u0442\u043e\u0441\u0430\u043b\u043e\u043d":"мотосалон",
   "visitvietnam":"visit vietnam",
   "hibachi":"hibachi",
   "anden":"anden",
   "hih":"hih",
   "staysafe":"stay safe",
   "vlogn\u1ea5u\u0103n":"vlog nau ăn",
   "hhennie":"hhennie",
   "bakugoucosplay":"bakugou cosplay",
   "fyp\u30b7\u309a":"fyp シャ",
   "dragonstaff":"dragon staff",
   "pov":"pov",
   "bikelife":"bike life",
   "organizewithme":"organize with me",
   "whipcrack":"whip crack",
   "parkourclips":"parkour clips",
   "tryitwithtiktok":"try it with tiktok",
   "reality":"reality",
   "\u043f\u0441\u0438\u0445":"псих",
   "deep

 97%|█████████▋| 33/34 [01:05<00:03,  3.55s/it]

{
  "ghoul":"ghoul",
   "10vancauhoivisao":"10 vạn câu hỏi vì sao",
   "shizoposting":"shizo posting",
   "sa":"sa",
   "sky":"sky",
   "firedance":"fire dance",
   "locktheclub":"lock the club",
   "scootertricks":"scooter tricks",
   "pubairlines":"pub airlines",
   "tradu\u00e7\u00e3o":"tradução",
   "\u0e27\u0e34\u0e2a\u0e01\u0e35\u0e49":"แม่เจ้า",
   "highschooldxd":"high school dxd",
   "punch":"punch",
   "studyaccount":"study account",
   "gacha":"gacha",
   "gopro":"go pro",
   "suhuong":"sư huong",
   "homedecor":"home decor",
   "sport":"sport",
   "\u1ea9mth\u1ef1c":"ấm thực",
   "domesticviolence":"domestic violence",
   "iamfine":"i am fine",
   "cry":"cry",
   "climbing":"climbing",
   "pyf":"pyf",
   "chocolate":"chocolate",
   "club":"club",
   "giftideas":"gift ideas",
   "hen":"hen",
   "papercraft":"paper craft",
   "jaypark_foryou":"jay park for you",
   "fup":"fup",
   "relatablememes":"relatable memes",
   "trongveo":"trống veo",
   "thethaomoingay":"thế thao mỗi

100%|██████████| 34/34 [01:08<00:00,  2.01s/it]

{
  "shesbroken":"she s broken",
   "h\u1ecdcn\u1ea5u\u0103n":"học nau ăn",
   "practise":"practise",
   "firegirl":"fire girl",
   "why":"why",
   "imnotok":"i m not ok",
   "ropedartdance":"rope dart dance",
   "viraltiktok":"viral tiktok",
   "depressed":"depressed",
   "restocking":"restocking",
   "motorcycle":"motorcycle",
   "1life":"1 life",
   "hoahau":"hoa hậu",
   "tiktokindia":"tiktok india",
   "teenager":"teenager",
   "nekedbe":"neked be",
   "iamsober":"i am sober",
   "depresao":"depresao",
   "cenangbeach":"cenang beach",
   "heartbroken":"heart broken",
   "fyppppppppppppppppppppppp":"fyppppppppppppppppppppppp",
   "husqvarna":"husqvarna",
   "aesthetic":"aesthetic",
   "unsaidfeelings":"unsaid feelings",
   "stunts":"stunts",
   "women":"women",
   "vba2023":"vba 2023",
   "ansiedad":"ansiedad",
   "drop":"drop",
   "aodaigoicam":"ao dai goi cam",
   "freefall":"free fall",
   "ohlaybrand":"oh lay brand",
   "whipcracking101":"whip cracking 101",
   "skydive":"skydi